# 과제 - 신경망을 이용한 손글씨 숫자 인식



## 1. 환경설정



In [1]:
# Colab: 이 셀을 가장 먼저 실행하세요 (저장소 클론 후 경로·모듈 로드)
# 주의: Colab에서는 GitHub 저장소 URL과 Personal Access Token을 반드시 입력해야 합니다.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from getpass import getpass

    git_url = input("GitHub 저장소 URL (예: github.com/USERNAME/mnist-lab.git): ").strip()
    token = getpass("GitHub Personal Access Token (private 저장소인 경우): ")

    # URL 마지막 경로를 저장소 폴더명으로 사용합니다. (예: .../mnist-lab.git -> mnist-lab)
    repo_name = Path(git_url.rstrip("/")).name
    if repo_name.endswith(".git"):
        repo_name = repo_name[:-4]

    !git clone https://{token}@{git_url}
    os.chdir(repo_name)
    sys.path.insert(0, str(Path.cwd() / "src"))
else:
    sys.path.insert(0, "./src")


GitHub 저장소 URL (예: github.com/USERNAME/mnist-lab.git): github.com/devhyun05/group4-mnist-lab
GitHub Personal Access Token (private 저장소인 경우): ··········
Cloning into 'group4-mnist-lab'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 49 (delta 12), reused 35 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 22.40 KiB | 11.20 MiB/s, done.
Resolving deltas: 100% (12/12), done.


## 2. 데이터 로드

In [2]:
from data import load_mnist

(x_train, y_train), (x_test, y_test) = load_mnist()
print('Train:', x_train.shape, y_train.shape)
print('Test:', x_test.shape, y_test.shape)

Train: (60000, 784) (60000,)
Test: (10000, 784) (10000,)


## 3. 구현 및 테스트 통과 확인

`src/` 아래 역할별 파일의 **TODO**를 순서대로 구현한 뒤 아래 셀을 실행하세요.
- 주요 구현 파일: `activations.py`, `layers.py`, `losses.py`, `optimizers.py`, `network.py`, `training.py`
- 구현 파일은 역할별 모듈을 직접 import합니다. 예: `from network import NeuralNetwork`
- 개발 순서: 과제 안내문 참조
- 테스트: `tests/` 아래의 단계별 단위 테스트를 필요한 파일부터 실행합니다. 처음에는 전체 테스트보다 맡은 부분의 테스트 파일을 먼저 실행하세요.
    - ReLU만 확인: `TEST_TARGET = "tests/test_relu.py"`
    - 파일 안의 일부 테스트만 확인: `PYTEST_KEYWORD = "backward"`
    - 전체 테스트 확인: `TEST_TARGET = "tests/"`

In [8]:
import subprocess
import sys
from pathlib import Path

# Colab/로컬 모두 현재 노트북 실행 위치를 저장소 루트로 사용합니다.
repo_dir = Path.cwd()

# 처음에는 자신이 구현 중인 부분의 테스트 파일만 실행하세요.
# 예: tests/test_relu.py, tests/test_affine.py, tests/test_training.py
TEST_TARGET = "tests/test_softmax.py"

# 특정 이름이 들어간 테스트만 실행하고 싶을 때 사용합니다.
# 예: "backward". 전체 파일을 실행하려면 빈 문자열로 둡니다.
PYTEST_KEYWORD = ""

cmd = [sys.executable, "-m", "pytest", TEST_TARGET, "-v"]
if PYTEST_KEYWORD:
    cmd.extend(["-k", PYTEST_KEYWORD])

print("실행 경로:", repo_dir)
print("실행 명령:", " ".join(cmd))
result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    cwd=str(repo_dir)
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode == 0:
    print("\n선택한 테스트를 통과했습니다.")
else:
    print("\n선택한 테스트 중 실패가 있습니다.")


실행 경로: /content/group4-mnist-lab
실행 명령: /usr/bin/python3 -m pytest tests/test_softmax.py -v
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/group4-mnist-lab
plugins: anyio-4.13.0, typeguard-4.5.1, langsmith-0.7.34
collecting ... collected 3 items

tests/test_softmax.py::TestSoftmax::test_softmax_forward_sum_one PASSED  [ 33%]
tests/test_softmax.py::TestSoftmax::test_softmax_forward_non_negative PASSED [ 66%]
tests/test_softmax.py::TestSoftmax::test_softmax_backward_shape PASSED   [100%]

============================== 3 passed in 0.12s ===============================


선택한 테스트를 통과했습니다.


## 4. 모델·옵티마이저 생성 및 학습

In [ ]:
from network import NeuralNetwork
from optimizers import Adam
from training import train

model = NeuralNetwork(use_batchnorm=True, use_dropout=True)  # BatchNorm, Dropout 필수
optimizer = Adam(lr=0.001)

loss_history = train(model, optimizer, x_train, y_train, epochs=20, batch_size=128)

## 5. 평가 및 손실 커브

In [ ]:
from training import evaluate, plot_loss_history

acc, n_params = evaluate(model, x_test, y_test)
print(f'Test Accuracy: {acc:.2f}%')
print(f'Total Params: {n_params:,}')

plot_loss_history(loss_history)

In [15]:
import os
from pathlib import Path

FOLDERNAME = 'mnist_lab'
# CORRECTED: Use the absolute path for Google Drive
FOLDER_TO_SAVE = os.path.join('/content/drive/My Drive/', FOLDERNAME)
FILES_TO_SAVE = ['group4-mnist-lab/src/activations.py']

repo_name = 'group4-mnist-lab' # Assuming repo_name is defined from earlier cell

for files in FILES_TO_SAVE:
  source_file_path_relative_to_cwd = Path(files).relative_to(repo_name)

  # Construct the full destination path in Google Drive
  destination_full_path = FOLDER_TO_SAVE / source_file_path_relative_to_cwd

  # Ensure all necessary directories for the destination file exist.
  destination_full_path.parent.mkdir(parents=True, exist_ok=True)

  # Now, open the source file and write its content to the destination file.
  with open(destination_full_path, 'w') as f:
    with open(source_file_path_relative_to_cwd, 'r') as source_f:
      f.write(source_f.read())